In [ ]:
!pip install pandas

In [ ]:
import requests
import pandas as pd
import time
import statsmodels.api as sm

# Preparing crypto coin data

In [ ]:
url = "https://api.binance.com/api/v3/klines"

pairs = [
   "BTCUSDT", "ETHUSDT", "XRPUSDT", "BCHUSDT", "LTCUSDT", "EOSUSDT", "BNBUSDT", "XTZUSDT", "QTUMUSDT", "RVNUSDT", "ZRXUSDT", "ALGOUSDT", "HOTUSDT", "THETAUSDT", "ZENUSDT", "KNCUSDT", "FETUSDT", "CHZUSDT", "FUNUSDT", "LRCUSDT", "HBARUSDT", "ADAUSDT", "DGBUSDT", "SNXUSDT", "ZILUSDT", "DCRUSDT", "ICXUSDT", "LSKUSDT", "ENJUSDT", "PAXGUSDT", "RLCUSDT", "RSRUSDT", "ARDRUSDT", "KAVAUSDT", "SXPUSDT", "TFUELUSDT", "DATAUSDT", "DOTUSDT", "LINKUSDT", "YFIUSDT", "COMPUSDT", "ZECUSDT", "SUSHIUSDT", "KSMUSDT", "UMAUSDT", "EGLDUSDT", "ONTUSDT", "LUNAUSDT", "STXUSDT", "BATUSDT", "AVAXUSDT", "NEARUSDT", "STORJUSDT", "ANKRUSDT", "CVCUSDT", "UTKUSDT", "WANUSDT", "JSTUSDT", "HIVEUSDT", "FTMUSDT", "IOTXUSDT", "DOGEUSDT", "ATOMUSDT", "NEOUSDT", "RUNEUSDT", "DASHUSDT", "VTHOUSDT", "DENTUSDT", "ONGUSDT", "OGNUSDT", "OXTUSDT", "BANDUSDT", "XVSUSDT", "SANDUSDT", "NMRUSDT", "CTSIUSDT", "CELRUSDT", "NKNUSDT", "SOLUSDT", "ETCUSDT", "AAVEUSDT", "ONEUSDT", "CRVUSDT", "IOSTUSDT", "CHRUSDT", "WINUSDT", "DUSKUSDT", "COTIUSDT", "FILUSDT", "BNTUSDT", "MANAUSDT", "SCUSDT", "VETUSDT", "TRXUSDT", "SUNUSDT"
]

interval = "1d"

start_date = "2019-10-01"
end_date = "2025-12-31"

start_timestamp = int(pd.Timestamp(start_date).timestamp() * 1000)
end_timestamp = int(pd.Timestamp(end_date).timestamp() * 1000)

all_pairs_data = []

for pair in pairs:

    print(f"Downloading {pair}...")

    start_time = start_timestamp

    all_data = []

    while start_time < end_timestamp:

        params = {
            "symbol": pair,
            "interval": interval,
            "startTime": start_time,
            "limit": 1000
        }

        data = requests.get(url, params=params).json()

        if len(data) == 0:
            break

        df = pd.DataFrame(data)

        all_data.append(df)

        start_time = data[-1][0] + 1

        time.sleep(0.2)

    pair_df = pd.concat(all_data, ignore_index=True)

    pair_df.columns = [
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "num_trades",
        "taker_buy_base",
        "taker_buy_quote",
        "ignore"
    ]

    pair_df["date"] = pd.to_datetime(
        pair_df["date"],
        unit="ms"
    )

    pair_df = pair_df[["date", "close"]]

    pair_df = pair_df[
        pair_df["date"] < "2026-01-01"
    ]

    pair_df["Pair"] = pair

    all_pairs_data.append(pair_df)

crypto_prices = pd.concat(
    all_pairs_data,
    ignore_index=True
)

In [ ]:
crypto_prices = crypto_prices.sort_values(
["Pair","date"]
)

In [ ]:
crypto_prices["close"] = (
crypto_prices["close"].astype(float)
)

In [ ]:
# --- convert USDT prices to USD ---
fx = pd.read_excel("USDT_USD price per day.xlsx")
fx = fx.rename(columns={"Date": "date", "Price": "usdt_usd"})   
fx["date"] = pd.to_datetime(fx["date"]).dt.normalize()
fx = fx[["date", "usdt_usd"]].sort_values("date")

crypto_prices["date"] = pd.to_datetime(crypto_prices["date"]).dt.normalize()   
crypto_prices = crypto_prices.merge(fx, on="date", how="left")
crypto_prices["close"] = crypto_prices["close"] * crypto_prices["usdt_usd"]     
crypto_prices = crypto_prices.drop(columns="usdt_usd")                         

In [ ]:
print("rows with no FX rate:", crypto_prices["close"].isna().sum())

In [ ]:
import numpy as np

crypto_prices["log_return"] = (
crypto_prices.groupby("Pair")["close"].transform(lambda x: np.log(x / x.shift(1)))
)

# Adding events and windows to the dataset

In [ ]:
events = [
    {"name": "SEC_charges_ripple", "date": "2020-12-22"},
    {"name": "SEC_charges_binance", "date": "2023-06-05"},
    {"name": "SEC_approves_BTC_ETF", "date": "2024-01-11"},
    {"name": "CFTC_Charges_BitMEX", "date": "2020-10-01"},
    {"name": "SEC_charges_Kraken", "date": "2023-02-09"},
    {"name": "CFTC/SEC_joint_statement_spot_crypto_trading", "date": "2025-09-02"},
    {"name": "BTC_not_safe_word_tweet", "date": "2020-01-10"},
    {"name": "Buy_tesla_with_BTC_tweet", "date": "2021-03-24"},
    {"name": "This_is_true_power_tweet", "date": "2021-02-10"},
    {"name": "Trump_tweet_america_crypto_capital", "date": "2024-09-30"},
    {"name": "Tesla_suspend_crypto_purchase_tweet", "date": "2021-05-13"},
    {"name": "ETH_transition_to_POS_tweet", "date": "2022-09-15"},
    {"name": "bybit_hack", "date": "2025-02-21"},
    {"name": "Ronin_bridge_hack", "date": "2022-03-29"},
    {"name": "polynetwork_hack", "date": "2021-08-10"},
    {"name": "Binance_bridge_hack", "date": "2022-10-06"},
    {"name": "Portal_attack", "date": "2022-02-02"},
    {"name": "DMM_Bitcoin_hack", "date": "2024-05-31"},
    {"name": "SVB_collapse", "date": "2023-03-08"},
    {"name": "US_tarrifs_announcement", "date": "2025-04-02"},
    {"name": "FED_announces_QE", "date": "2020-03-23"},
    {"name": "ECB_increase_main_IR", "date": "2022-07-21"},
    {"name": "FED_increase_primary_credit_rate", "date": "2022-06-15"},
    {"name": "Bank_of_japan_end_negative_rates", "date": "2024-03-19"},
]

crypto_prices["date"] = pd.to_datetime(crypto_prices["date"]).dt.normalize()

EST_START, EST_END = 90, 10   # estimation window: event-90 ... event-10
EVT_PRE, EVT_POST  = 2, 4     # event window:      event-2  ... event+5

events_sorted = sorted(events, key=lambda e: pd.Timestamp(e["date"]))

# --- Pass 1: event windows are protected and collected globally ---
event_windows = {}
all_event_window_dates = set()
for ev in events_sorted:
    d = pd.Timestamp(ev["date"])
    w = set(pd.date_range(d - pd.Timedelta(days=EVT_PRE),
                          d + pd.Timedelta(days=EVT_POST), freq="D"))
    event_windows[ev["name"]] = w
    clash = all_event_window_dates & w
    if clash:
        print(f"[warn] {ev['name']} event window overlaps another event window: {sorted(clash)}")
    all_event_window_dates |= w

# --- Pass 2: estimation windows, excluding any event window (of any event)
#     and any day already taken by an earlier event's estimation window ---
estimation_windows = {}
claimed_estimation_dates = set()
for ev in events_sorted:
    d = pd.Timestamp(ev["date"])
    full = set(pd.date_range(d - pd.Timedelta(days=EST_START),
                             d - pd.Timedelta(days=EST_END), freq="D"))
    clean = full - all_event_window_dates - claimed_estimation_dates
    estimation_windows[ev["name"]] = clean
    claimed_estimation_dates |= clean

# --- Write the columns (0 = outside, 1 = estimation, 2 = event) ---
for ev in events:
    name = ev["name"]
    crypto_prices[name] = 0
    crypto_prices.loc[crypto_prices["date"].isin(estimation_windows[name]), name] = 1
    crypto_prices.loc[crypto_prices["date"].isin(event_windows[name]),      name] = 2

# --- Summary ---
for ev in events_sorted:
    n = len(estimation_windows[ev["name"]])
    print(f"{ev['name']:<48} estimation days: {n:>2}  (event {ev['date']})")

# Excluding events with low number of observations

In [ ]:
MIN_EST_DAYS = 30

kept, dropped = [], []
for ev in events_sorted:
    n = len(estimation_windows[ev["name"]])
    (dropped if n < MIN_EST_DAYS else kept).append((ev["name"], ev["date"], n))

print(f"DROPPED (estimation days < {MIN_EST_DAYS}):")
for name, date, n in dropped:
    print(f"  {name:<48} {date}  days={n}")

print(f"\nKEPT ({len(kept)} events):")
for name, date, n in kept:
    print(f"  {name:<48} {date}  days={n}")

In [ ]:
MIN_EST_DAYS_LIMIT = 30 

events_to_remove = [name for name, days in estimation_windows.items()
                    if len(days) < MIN_EST_DAYS]

print("Dropping:", events_to_remove)
crypto_prices = crypto_prices.drop(columns=events_to_remove, errors="ignore")

# keep your events list in sync so later loops don't reference dropped events
events = [ev for ev in events if ev["name"] not in events_to_remove]

# Removing extreme observations

In [ ]:
to_drop = [
    ("2022-05-31", "LUNAUSDT"),
    ("2022-05-12", "LUNAUSDT"),
    ("2021-06-18", "SUNUSDT"),
    ("2022-05-11", "LUNAUSDT"),
    ("2022-05-13", "LUNAUSDT"),
]
drop_keys = {(pd.Timestamp(d).normalize(), p) for d, p in to_drop}

before = len(crypto_prices)
mask = [ (d, p) in drop_keys for d, p in zip(crypto_prices["date"], crypto_prices["Pair"]) ]
crypto_prices = crypto_prices[~pd.Series(mask, index=crypto_prices.index)].copy()

print(f"dropped {before - len(crypto_prices)} rows (expected 5)")

# Calculating abnormal returns

In [ ]:
events_df = pd.DataFrame(events)
event_columns = events_df["name"].tolist()

MIN_EST_OBS = 20          # skip a coin if it has too few estimation-window returns

all_results = []

for event in event_columns:

    # --- ESTIMATION WINDOW: per-coin mean (and sd) of log returns ---
    estimation_data = crypto_prices.loc[
        crypto_prices[event] == 1,
        ["date", "Pair", "log_return"]
    ].dropna()

    est_stats = (
        estimation_data
        .groupby("Pair")["log_return"]
        .agg(mean_est="mean", sd_est="std", n_est="count")
    )

    # keep only coins with enough estimation observations
    est_stats = est_stats[est_stats["n_est"] >= MIN_EST_OBS]

    # --- EVENT WINDOW: actual returns per coin/date ---
    event_data = crypto_prices.loc[
        crypto_prices[event] == 2,
        ["date", "Pair", "log_return"]
    ].dropna()

    # only coins that have BOTH a valid estimation mean and event-window data
    common_pairs = event_data["Pair"].unique()
    common_pairs = [p for p in common_pairs if p in est_stats.index]

    for pair in common_pairs:

        mean_est = est_stats.loc[pair, "mean_est"]
        sd_est   = est_stats.loc[pair, "sd_est"]
        n_est    = est_stats.loc[pair, "n_est"]

        temp = event_data.loc[event_data["Pair"] == pair, ["date", "log_return"]].copy()
        temp = temp.rename(columns={"log_return": "actual_return"})

        temp["expected_return"] = mean_est                          # mean-adjusted: E[R] = mean
        temp["abnormal_return"] = temp["actual_return"] - mean_est

        temp["event"]   = event
        temp["pair"]    = pair
        temp["mean_est"] = mean_est
        temp["sd_est"]   = sd_est      # sd of estimation returns -> for test statistics later
        temp["n_est"]    = n_est

        temp = temp[
            ["event", "date", "pair",
             "actual_return", "expected_return", "abnormal_return",
             "mean_est", "sd_est", "n_est"]
        ]

        all_results.append(temp)

# --- FINAL MASTER TABLE ---
master_ar_table = pd.concat(all_results, ignore_index=True)

print(master_ar_table.head())

# Adding coin and event mapping

In [ ]:
# --- Event-level group (unchanged: this one really is event -> group) ---
master_ar_table["group"] = master_ar_table["event"].map({
    "SVB_collapse": "Macro",
    "US_tarrifs_announcement": "Macro",
    "FED_announces_QE": "Macro",
    "ECB_increase_main_IR": "Macro",
    "FED_increase_primary_credit_rate": "Macro",
    "Bank_of_japan_end_negative_rates": "Macro",
    "SEC_charges_ripple": "Regulatory",
    "SEC_charges_binance": "Regulatory",
    "SEC_approves_BTC_ETF": "Regulatory",
    "CFTC_Charges_BitMEX": "Regulatory",
    "SEC_charges_Kraken": "Regulatory",
    "CFTC/SEC_joint_statement_spot_crypto_trading": "Regulatory",
    "bybit_hack": "Security & Exchange hacks",
    "Ronin_bridge_hack": "Security & Exchange hacks",
    "polynetwork_hack": "Security & Exchange hacks",
    "Binance_bridge_hack": "Security & Exchange hacks",
    "Portal_attack": "Security & Exchange hacks",
    "DMM_Bitcoin_hack": "Security & Exchange hacks",
    "BTC_not_safe_word_tweet": "Social postings",
    "Buy_tesla_with_BTC_tweet": "Social postings",
    "This_is_true_power_tweet": "Social postings",
    "Trump_tweet_america_crypto_capital": "Social postings",
    "Tesla_suspend_crypto_purchase_tweet": "Social postings",
    "ETH_transition_to_POS_tweet": "Social postings",
})

# --- Coin group + expected direction: JOIN by event AND pair ---
coin_map = pd.read_excel("Mapping table for Coins.xlsx")   # adjust path/filename

# align key column names to master_ar_table's ("event", "pair")
coin_map = coin_map.rename(columns={
    "Event": "event",
    "Pair": "pair",
    "Coin group": "coin_group",
    "expected direction": "expected_direction",
})
coin_map = coin_map[["event", "pair", "coin_group", "expected_direction"]]

master_ar_table = master_ar_table.merge(
    coin_map,
    on=["event", "pair"],
    how="left",          # keeps every AR row; unmapped coins get NaN coin_group
)

# --- sanity check: which AR rows didn't get a coin group? ---
unmatched = master_ar_table[master_ar_table["coin_group"].isna()]
print(f"AR rows without a coin group: {len(unmatched)}")
if len(unmatched):
    print(unmatched[["event", "pair"]].drop_duplicates().to_string(index=False))

# Calculating signed/absolute SAR per each day in the window

In [ ]:
import numpy as np
import pandas as pd

# master_ar_table (RQ3 version): one row per event-coin-DAY, with
# event, pair, date, abnormal_return, mean_est, sd_est, n_est, expected_direction, ...

m = master_ar_table.copy()
m = m[(m["sd_est"] > 0) & (m["n_est"] >= 20) & m["expected_direction"].notna()]          # usable coin-events only

# relative day (-2 ... +4) from the event date
event_dates = {e["name"]: pd.Timestamp(e["date"]) for e in events}
m["rel_day"] = [ (d - event_dates[ev]).days for d, ev in zip(m["date"], m["event"]) ]

# single-day standardized, signed abnormal return
m["signed_SAR"] = (m["abnormal_return"] / m["sd_est"]) * m["expected_direction"]
m["abs_SAR"] = (m["abnormal_return"] / m["sd_est"]).abs()

# OBSERVED: mean over coins within event, then mean over events -> one number per rel_day
observed_by_day = (m.groupby(["rel_day", "event"])["signed_SAR"].mean()   # within event
                     .groupby("rel_day").mean())                          # across events
observed_by_day_abs = (m.groupby(["rel_day", "event"])["abs_SAR"].mean()   # within event
                         .groupby("rel_day").mean())
print(observed_by_day)

# Calculating estimation window null and p-values per day - signed SAR

In [ ]:
# per event-coin constants (one row each)
const = (m.drop_duplicates(["event", "pair"])
           .set_index(["event", "pair"])[["mean_est", "sd_est", "expected_direction"]])

B = 10000
rng = np.random.default_rng(42)

# For each event, precompute R_e: within-event mean (over coins) of the signed standardized
# abnormal return, for every estimation-window day.  Drawing one value from R_e = drawing one day.
event_R = {}
for event, grp in const.groupby(level=0):
    pairs    = grp.index.get_level_values(1).tolist()
    mean_est = grp["mean_est"].values
    sd_est   = grp["sd_est"].values
    dirs     = grp["expected_direction"].values

    est = (crypto_prices[(crypto_prices["Pair"].isin(pairs)) & (crypto_prices[event] == 1)]
           .pivot_table(index="date", columns="Pair", values="log_return")
           .sort_index()
           .reindex(columns=pairs))                     # align columns to const order

    Z = (est.values - mean_est) / sd_est * dirs         # signed standardized AR, days x coins
    R = np.nanmean(Z, axis=1)                            # within-event mean over coins, per day
    R = R[~np.isnan(R)]
    if len(R) >= 20:
        event_R[event] = R

events_used = list(event_R.keys())

# each replication: one random estimation day per event, averaged across events
draws = np.column_stack([rng.choice(event_R[ev], size=B, replace=True) for ev in events_used])
null_dist = draws.mean(axis=1)                           # length B: the common null

In [ ]:
rows = []
for rel_day, obs in observed_by_day.items():
    p_one = (np.sum(null_dist >= obs) + 1) / (B + 1)     # one-sided: moved in expected direction
    rows.append([rel_day, obs, p_one])

result = (pd.DataFrame(rows, columns=["rel_day", "observed_signed_SAR", "p_value"])
            .sort_values("rel_day").reset_index(drop=True))
print(result)

# Calculating estimation window null and p-values per day per event - signed SAR

In [ ]:
B = 10000
rng = np.random.default_rng(42)

rows = []
for event, gc in const.groupby(level=0):
    # ---- observed: mean over coins per relative day, for THIS event only
    obs_day = (m[m["event"] == event]
               .groupby("rel_day")["signed_SAR"].mean())

    # ---- null: within-event mean signed standardized AR, per estimation day
    pairs    = gc.index.get_level_values(1).tolist()
    mean_est = gc["mean_est"].values
    sd_est   = gc["sd_est"].values
    dirs     = gc["expected_direction"].values

    est = (crypto_prices[(crypto_prices["Pair"].isin(pairs)) & (crypto_prices[event] == 1)]
           .pivot_table(index="date", columns="Pair", values="log_return")
           .sort_index().reindex(columns=pairs))

    Z = (est.values - mean_est) / sd_est * dirs        # days x coins  (same day across coins in a row)
    R = np.nanmean(Z, axis=1)                           # one value per estimation day
    R = R[~np.isnan(R)]
    if len(R) < 20:
        continue

    null_dist = rng.choice(R, size=B, replace=True)     # each pull = one estimation day, all coins together

    for rel_day, obs in obs_day.items():
        p_one = (np.sum(null_dist >= obs) + 1) / (B + 1)
        rows.append([event, rel_day, obs, p_one])

per_event_result = (pd.DataFrame(rows, columns=["event", "rel_day", "observed_signed_SAR", "p_value"])
                    .sort_values(["event", "rel_day"]).reset_index(drop=True))
print(per_event_result.head(16))

# Calculating estimation window null and p-values per day - absolute SAR

In [ ]:
const = (m.drop_duplicates(["event", "pair"])
           .set_index(["event", "pair"])[["mean_est", "sd_est", "expected_direction"]])

B = 10000
rng = np.random.default_rng(42)

event_R_abs = {}
for event, grp in const.groupby(level=0):
    pairs    = grp.index.get_level_values(1).tolist()
    mean_est = grp["mean_est"].values
    sd_est   = grp["sd_est"].values

    est = (crypto_prices[(crypto_prices["Pair"].isin(pairs)) & (crypto_prices[event] == 1)]
           .pivot_table(index="date", columns="Pair", values="log_return")
           .sort_index().reindex(columns=pairs))

    Z_abs = np.abs((est.values - mean_est) / sd_est)    # |standardized AR|, days x coins (abs at coin level)
    R = np.nanmean(Z_abs, axis=1)                         # within-event mean |SAR| per estimation day
    R = R[~np.isnan(R)]
    if len(R) >= 20:
        event_R_abs[event] = R

events_used = list(event_R_abs.keys())
draws = np.column_stack([rng.choice(event_R_abs[ev], size=B, replace=True) for ev in events_used])
null_dist_abs = draws.mean(axis=1)

rows = []
for rel_day, obs in observed_by_day_abs.items():
    p_one = (np.sum(null_dist_abs >= obs) + 1) / (B + 1)   # moved MORE in magnitude than noise
    rows.append([rel_day, obs, p_one])

result_abs = (pd.DataFrame(rows, columns=["rel_day", "observed_abs_SAR", "p_value"])
                .sort_values("rel_day").reset_index(drop=True))
print(result_abs)

# Calculating estimation window null and p-values per day per event - absolute SAR

In [ ]:
B = 10000
rng = np.random.default_rng(42)

rows = []
for event, gc in const.groupby(level=0):
    # observed: mean over coins per rel_day, this event only
    obs_day = (m[m["event"] == event]
               .groupby("rel_day")["abs_SAR"].mean())

    pairs    = gc.index.get_level_values(1).tolist()
    mean_est = gc["mean_est"].values
    sd_est   = gc["sd_est"].values

    est = (crypto_prices[(crypto_prices["Pair"].isin(pairs)) & (crypto_prices[event] == 1)]
           .pivot_table(index="date", columns="Pair", values="log_return")
           .sort_index().reindex(columns=pairs))

    Z_abs = np.abs((est.values - mean_est) / sd_est)    # |standardized AR|, days x coins
    R = np.nanmean(Z_abs, axis=1)
    R = R[~np.isnan(R)]
    if len(R) < 20:
        continue

    null_dist_ev = rng.choice(R, size=B, replace=True)

    for rel_day, obs in obs_day.items():
        p_one = (np.sum(null_dist_ev >= obs) + 1) / (B + 1)
        rows.append([event, rel_day, obs, p_one])

per_event_result_abs = (pd.DataFrame(rows, columns=["event", "rel_day", "observed_abs_SAR", "p_value"])
                        .sort_values(["event", "rel_day"]).reset_index(drop=True))
print(per_event_result_abs.head(16))

In [ ]:
result.to_excel("per_day_signed.xlsx")
result_abs.to_excel("per_day_abs.xlsx")
per_event_result.to_excel("per_event_signed.xlsx")
per_event_result_abs.to_excel("per_event_abs.xlsx")